# Explore critical-word probabilities with Meltemi

Loads the model once, scores the critical word in each stimulus row against its preceding context, and compares conditions within each item.

Approach: tokenize the *full* sentence once (not context and critical word separately), then use the tokenizer's offset mapping to find which tokens belong to the critical word. This avoids boundary retokenization issues that come from concatenating separately tokenized strings.

> [!NOTE]
> In Colab, code preceded with `!` is run as bash/shell, not Python.

### Push to GitHub repo

Locate the 'Save in GitHub to keep changes' button, usually up beside the file name (top left).

### Import modules and load model

> [!Note:]
> `hf_device_map` may not exist if the whole model fit on one device (expected on T4)

In [3]:
# for missing packages install via bash/shell
!pip install -U bitsandbytes
# then Runtime > Restart session to refresh kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.0 MB/s eta 0:00:00


In [1]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "ilsp/Meltemi-7B-v1.5"
STIMULI_PATH = "stimuli/stimuli_demo.csv"

In [2]:
# Load model and tokenizer once. This is the slow part, run it once per session.
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
model.eval()

print(f"Model loaded, device map: {model.hf_device_map}")

config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.21k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.97M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.18MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

AttributeError: 'MistralForCausalLM' object has no attribute 'hf_device_map'

## Scoring function

Given a full sentence and the critical word's character span within it, returns:
- `logprob`: summed log probability of the critical word's tokens, conditioned on everything before it
- `surprisal`: -log2(probability), in bits
- `probability`: exp(logprob), the raw probability (useful as the predictability / cloze-proxy measure)

In [3]:
import math

def score_critical_word(full_sentence, critical_word, tokenizer, model, device):
    # locate the critical word's character span in the full sentence
    # use rfind (not find) since for antecedent-QA scoring, the candidate (e.g. "O gátos")
    # may also appear earlier in the sentence, and we want the LAST occurrence (the actual answer span)
    start_char = full_sentence.rfind(critical_word)
    if start_char == -1:
        raise ValueError(f"Critical word '{critical_word}' not found in sentence: {full_sentence}")
    end_char = start_char + len(critical_word)

    encoding = tokenizer(full_sentence, return_offsets_mapping=True, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    offsets = encoding["offset_mapping"][0]

    # find which token indices fall inside the critical word's character span
    critical_token_idxs = [
        i for i, (s, e) in enumerate(offsets.tolist())
        if s < end_char and e > start_char and not (s == 0 and e == 0)  # skip special tokens
    ]
    if not critical_token_idxs:
        raise ValueError(f"No tokens matched the critical word span for: {critical_word}")

    with torch.no_grad():
        outputs = model(input_ids)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)

    # log P(token_i | tokens_<i) comes from the logits at position i-1
    total_logprob = 0.0
    for idx in critical_token_idxs:
        token_id = input_ids[0, idx]
        total_logprob += log_probs[idx - 1, token_id].item()

    return {
        "logprob": total_logprob,
        "surprisal_bits": -total_logprob / math.log(2),
        "probability": math.exp(total_logprob),
        "n_tokens": len(critical_token_idxs),
    }

## Quick sanity check on a single item

Before running the full stimuli set, confirm the function behaves sensibly on one hand-picked example.

In [4]:
device = next(model.parameters()).device
test_cases = [
    {"pronoun": "null", "condition": "neutral", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"pronoun": "null", "condition": "subject-preferred", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"pronoun": "null", "condition": "object-preferred", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
    {"pronoun": "overt", "condition": "neutral", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"pronoun": "overt", "condition": "subject-preferred", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"pronoun": "overt", "condition": "object-preferred", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
]

base_null = "O gátos íde ton peteinó ótan épsachne gia {word}."
base_overt = "O gátos íde ton peteinó ótan aftós épsachne gia {word}."

results2 = []
for case in test_cases:
    sentence = (base_null if case["pronoun"] == "null" else base_overt).format(word=case["critical_word"])
    r = score_critical_word(sentence, case["critical_word"], tokenizer, model, device)
    results2.append({
        "pronoun": case["pronoun"],
        "condition": case["condition"],
        "world_knowledge": case["world_knowledge"],
        "critical_word": case["critical_word"],
        "logprob": r["logprob"],
        "surprisal_bits": r["surprisal_bits"],
        "probability": r["probability"],
        "n_tokens": r["n_tokens"],
    })

df2 = pd.DataFrame(results2)
df2

,pronoun,condition,world_knowledge,critical_word,logprob,surprisal_bits,probability,n_tokens
0,null,neutral,neutral,fagitó,-9.039062,13.040611,1.186820e-04,4
1,null,subject-preferred,match,pontíkia,-14.722656,21.240303,4.036748e-07,4
2,null,object-preferred,mismatch,skoulíkia,-15.957031,23.021130,1.174761e-07,5
3,overt,neutral,neutral,fagitó,-8.980469,12.956078,1.258438e-04,4
4,overt,subject-preferred,match,pontíkia,-14.359375,20.716199,5.805006e-07,4
5,overt,object-preferred,mismatch,skoulíkia,-16.710938,24.108787,5.527542e-08,5


In [5]:
test_cases = [
    {"word_order": "original", "pronoun": "null", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"word_order": "original", "pronoun": "null", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"word_order": "original", "pronoun": "null", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
    {"word_order": "original", "pronoun": "overt", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"word_order": "original", "pronoun": "overt", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"word_order": "original", "pronoun": "overt", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
    {"word_order": "swapped", "pronoun": "null", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"word_order": "swapped", "pronoun": "null", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"word_order": "swapped", "pronoun": "null", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
    {"word_order": "swapped", "pronoun": "overt", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"word_order": "swapped", "pronoun": "overt", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"word_order": "swapped", "pronoun": "overt", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
]

templates = {
    "original": {
        "null": "O gátos íde ton peteinó ótan épsachne gia {word}.",
        "overt": "O gátos íde ton peteinó ótan aftós épsachne gia {word}.",
        "subject": "O gátos", "object": "O peteinós",
    },
    "swapped": {
        "null": "O peteinós íde ton gáto ótan épsachne gia {word}.",
        "overt": "O peteinós íde ton gáto ótan aftós épsachne gia {word}.",
        "subject": "O peteinós", "object": "O gátos",
    },
}

In [6]:
def score_antecedent_full(context, question, candidates, tokenizer, model, device):
    """
    Score each candidate answer to a comprehension question, given a context sentence.
    Returns the full score dict (probability, surprisal, logprob, n_tokens) per candidate,
    so we can compare not just probability but also token count (candidates may tokenize differently).
    """
    results = {}
    for candidate in candidates:
        full_text = f"{context} {question} {candidate}"
        r = score_critical_word(full_text, candidate, tokenizer, model, device)
        results[candidate] = r
    return results


results4 = []
for case in test_cases:
    # pick the right sentence template based on word order (original vs subject/object swapped)
    tmpl = templates[case["word_order"]]

    # build the actual context sentence (null or overt pronoun) with the critical word inserted
    context = tmpl[case["pronoun"]].format(word=case["critical_word"])

    # comprehension question asking who performed the action tied to the critical word
    question = f"Piós épsachne gia {case['critical_word']}?"

    # candidates are the two possible antecedents: whichever NP is subject vs object in this word order
    candidates = [tmpl["subject"], tmpl["object"]]

    # score both candidates as continuations of context + question
    scores = score_antecedent_full(context, question, candidates, tokenizer, model, device)

    subj_scores = scores[tmpl["subject"]]
    obj_scores = scores[tmpl["object"]]

    results4.append({
        "word_order": case["word_order"],       # original vs swapped NP order
        "pronoun": case["pronoun"],              # null vs overt
        "world_knowledge": case["world_knowledge"],  # match / neutral / mismatch
        "critical_word": case["critical_word"],

        # which NP is grammatical subject/object in this word order (varies when swapped)
        "subject": tmpl["subject"],
        "object": tmpl["object"],

        # probability the model assigns to each candidate as the answer
        "p_subject": subj_scores["probability"],
        "p_object": obj_scores["probability"],

        # surprisal (bits) as an alternate scale, useful for plotting/stats since it's roughly linear
        "subject_surprisal_bits": subj_scores["surprisal_bits"],
        "object_surprisal_bits": obj_scores["surprisal_bits"],

        # raw log probability (natural log), kept for reference/debugging
        "subject_logprob": subj_scores["logprob"],
        "object_logprob": obj_scores["logprob"],

        # token count per candidate, worth checking since subject/object names may tokenize differently
        # (e.g. different lengths could bias probability comparisons)
        "subject_n_tokens": subj_scores["n_tokens"],
        "object_n_tokens": obj_scores["n_tokens"],
    })

df4 = pd.DataFrame(results4)
df4

,word_order,pronoun,world_knowledge,critical_word,subject,object,p_subject,p_object,subject_surprisal_bits,object_surprisal_bits,subject_logprob,object_logprob,subject_n_tokens,object_n_tokens
0,original,null,match,pontíkia,O gátos,O peteinós,0.015542,0.008373,6.007692,6.899999,-4.164215,-4.782715,4,4
1,original,null,neutral,fagitó,O gátos,O peteinós,0.010714,0.016077,6.544344,5.958866,-4.536194,-4.130371,4,4
2,original,null,mismatch,skoulíkia,O gátos,O peteinós,0.011813,0.008179,6.403500,6.933812,-4.438568,-4.806152,4,4
3,original,overt,match,pontíkia,O gátos,O peteinós,0.018368,0.015639,5.766686,5.998667,-3.997162,-4.157959,4,4
4,original,overt,neutral,fagitó,O gátos,O peteinós,0.015458,0.029261,6.015485,5.094869,-4.169617,-3.531494,4,4
5,original,overt,mismatch,skoulíkia,O gátos,O peteinós,0.015045,0.017010,6.054582,5.877503,-4.196716,-4.073975,4,4
6,swapped,null,match,pontíkia,O peteinós,O gátos,0.033491,0.004659,4.900091,7.745867,-3.396484,-5.369026,4,4
7,swapped,null,neutral,fagitó,O peteinós,O gátos,0.032153,0.013623,4.958912,6.197847,-3.437256,-4.296021,4,4
8,swapped,null,mismatch,skoulíkia,O peteinós,O gátos,0.034640,0.003056,4.851397,8.354339,-3.362732,-5.790787,4,4
9,swapped,overt,match,pontíkia,O peteinós,O gátos,0.039607,0.011094,4.658116,6.494054,-3.228760,-4.501335,4,4


In [8]:
def show_top_predictions(full_sentence, critical_word, tokenizer, model, device, top_k=10):
    # cut the sentence right before the critical word, so we predict from there
    start_char = full_sentence.find(critical_word)
    context = full_sentence[:start_char].rstrip()

    inputs = tokenizer(context, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]  # prediction for the very next token
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, top_k)
    predictions = [(tokenizer.decode([tid]), p.item()) for tid, p in zip(top_ids, top_probs)]
    return predictions


def generate_continuation(full_sentence, critical_word, tokenizer, model, device, max_new_tokens=8):
    start_char = full_sentence.find(critical_word)
    context = full_sentence[:start_char].rstrip()

    inputs = tokenizer(context, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # only decode the newly generated tokens, not the input context
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [9]:
results5 = []
for case in test_cases:
    tmpl = templates[case["word_order"]]
    context = tmpl[case["pronoun"]].format(word=case["critical_word"])

    # score the critical word itself (predictability measure)
    critical_scores = score_critical_word(context, case["critical_word"], tokenizer, model, device)

    # score the two candidate antecedents via the comprehension question
    question = f"Piós épsachne gia {case['critical_word']}?"
    candidates = [tmpl["subject"], tmpl["object"]]
    antecedent_scores = score_antecedent_full(context, question, candidates, tokenizer, model, device)

    subj_scores = antecedent_scores[tmpl["subject"]]
    obj_scores = antecedent_scores[tmpl["object"]]

    # what would the model actually put here? top-5 next-token predictions
    top_preds = show_top_predictions(context, case["critical_word"], tokenizer, model, device, top_k=5)
    top_preds_str = "; ".join(f"{tok!r}:{p:.3f}" for tok, p in top_preds)

    # free-generation continuation (greedy), for a human-readable sense check
    completion = generate_continuation(context, case["critical_word"], tokenizer, model, device, max_new_tokens=8)

    results5.append({
        "word_order": case["word_order"],
        "pronoun": case["pronoun"],
        "world_knowledge": case["world_knowledge"],
        "critical_word": case["critical_word"],

        # critical word predictability (main measure)
        "p_critical": critical_scores["probability"],
        "critical_surprisal_bits": critical_scores["surprisal_bits"],
        "critical_logprob": critical_scores["logprob"],
        "critical_n_tokens": critical_scores["n_tokens"],

        # antecedent identity in this word order
        "subject": tmpl["subject"],
        "object": tmpl["object"],

        # antecedent-QA probe (diagnostic only, see notebook note above)
        "p_subject": subj_scores["probability"],
        "p_object": obj_scores["probability"],
        "subject_surprisal_bits": subj_scores["surprisal_bits"],
        "object_surprisal_bits": obj_scores["surprisal_bits"],

        # what the model would actually generate instead of the critical word
        "top5_predictions": top_preds_str,
        "greedy_completion": completion,
    })

df5 = pd.DataFrame(results5)
df5

,word_order,pronoun,world_knowledge,critical_word,p_critical,critical_surprisal_bits,critical_logprob,critical_n_tokens,subject,object,p_subject,p_object,subject_surprisal_bits,object_surprisal_bits,top5_predictions,greedy_completion
0,original,null,match,pontíkia,4.036748e-07,21.240303,-14.722656,4,O gátos,O peteinós,0.015542,0.008373,6.007692,6.899999,'to':0.143; 'ton':0.126; 't':0.082; 'th':0.039...,to péton.\nO g
1,original,null,neutral,fagitó,1.186820e-04,13.040611,-9.039062,4,O gátos,O peteinós,0.010714,0.016077,6.544344,5.958866,'to':0.143; 'ton':0.126; 't':0.082; 'th':0.039...,to péton.\nO g
2,original,null,mismatch,skoulíkia,1.174761e-07,23.021130,-15.957031,5,O gátos,O peteinós,0.011813,0.008179,6.403500,6.933812,'to':0.143; 'ton':0.126; 't':0.082; 'th':0.039...,to péton.\nO g
3,original,overt,match,pontíkia,5.805006e-07,20.716199,-14.359375,4,O gátos,O peteinós,0.018368,0.015639,5.766686,5.998667,'to':0.136; 'ton':0.120; 'na':0.093; 't':0.093...,to péton.\nEN:
4,original,overt,neutral,fagitó,1.258438e-04,12.956078,-8.980469,4,O gátos,O peteinós,0.015458,0.029261,6.015485,5.094869,'to':0.136; 'ton':0.120; 'na':0.093; 't':0.093...,to péton.\nEN:
5,original,overt,mismatch,skoulíkia,5.527542e-08,24.108787,-16.710938,5,O gátos,O peteinós,0.015045,0.017010,6.054582,5.877503,'to':0.136; 'ton':0.120; 'na':0.093; 't':0.093...,to péton.\nEN:
6,swapped,null,match,pontíkia,2.580992e-07,21.885571,-15.169922,4,O peteinós,O gátos,0.033491,0.004659,4.900091,7.745867,'to':0.111; 't':0.086; 'th':0.056; 'm':0.052; ...,to spiti\nO peteinós
7,swapped,null,neutral,fagitó,1.135964e-03,9.781867,-6.780273,4,O peteinós,O gátos,0.032153,0.013623,4.958912,6.197847,'to':0.111; 't':0.086; 'th':0.056; 'm':0.052; ...,to spiti\nO peteinós
8,swapped,null,mismatch,skoulíkia,2.446022e-07,21.963060,-15.223633,5,O peteinós,O gátos,0.034640,0.003056,4.851397,8.354339,'to':0.111; 't':0.086; 'th':0.056; 'm':0.052; ...,to spiti\nO peteinós
9,swapped,overt,match,pontíkia,5.208658e-07,20.872585,-14.467773,4,O peteinós,O gátos,0.039607,0.011094,4.658116,6.494054,'to':0.111; 'na':0.081; 't':0.072; 'm':0.059; ...,to faghtó tou.\n


In [10]:
results6 = []
for case in test_cases:
    tmpl = templates[case["word_order"]]
    context = tmpl[case["pronoun"]].format(word=case["critical_word"])

    # critical word predictability (main measure)
    crit = score_critical_word(context, case["critical_word"], tokenizer, model, device)

    # antecedent probabilities: does the model treat subject or object as the referent,
    # after reading the whole sentence (context + comprehension question)
    question = f"Piós épsachne gia {case['critical_word']}?"
    subj = score_critical_word(f"{context} {question} {tmpl['subject']}", tmpl['subject'], tokenizer, model, device)
    obj = score_critical_word(f"{context} {question} {tmpl['object']}", tmpl['object'], tokenizer, model, device)

    results6.append({
        "word_order": case["word_order"],
        "pronoun": case["pronoun"],
        "world_knowledge": case["world_knowledge"],
        "critical_word": case["critical_word"],
        "subject": tmpl["subject"],
        "object": tmpl["object"],

        # critical word scores
        "crit_probability": crit["probability"],
        "crit_surprisal_bits": crit["surprisal_bits"],
        "crit_logprob": crit["logprob"],
        "crit_n_tokens": crit["n_tokens"],

        # subject-as-referent scores
        "subj_probability": subj["probability"],
        "subj_surprisal_bits": subj["surprisal_bits"],
        "subj_logprob": subj["logprob"],
        "subj_n_tokens": subj["n_tokens"],

        # object-as-referent scores
        "obj_probability": obj["probability"],
        "obj_surprisal_bits": obj["surprisal_bits"],
        "obj_logprob": obj["logprob"],
        "obj_n_tokens": obj["n_tokens"],
    })

df6 = pd.DataFrame(results6)
df6

,word_order,pronoun,world_knowledge,critical_word,subject,object,crit_probability,crit_surprisal_bits,crit_logprob,crit_n_tokens,subj_probability,subj_surprisal_bits,subj_logprob,subj_n_tokens,obj_probability,obj_surprisal_bits,obj_logprob,obj_n_tokens
0,original,null,match,pontíkia,O gátos,O peteinós,4.036748e-07,21.240303,-14.722656,4,0.015542,6.007692,-4.164215,4,0.008373,6.899999,-4.782715,4
1,original,null,neutral,fagitó,O gátos,O peteinós,1.186820e-04,13.040611,-9.039062,4,0.010714,6.544344,-4.536194,4,0.016077,5.958866,-4.130371,4
2,original,null,mismatch,skoulíkia,O gátos,O peteinós,1.174761e-07,23.021130,-15.957031,5,0.011813,6.403500,-4.438568,4,0.008179,6.933812,-4.806152,4
3,original,overt,match,pontíkia,O gátos,O peteinós,5.805006e-07,20.716199,-14.359375,4,0.018368,5.766686,-3.997162,4,0.015639,5.998667,-4.157959,4
4,original,overt,neutral,fagitó,O gátos,O peteinós,1.258438e-04,12.956078,-8.980469,4,0.015458,6.015485,-4.169617,4,0.029261,5.094869,-3.531494,4
5,original,overt,mismatch,skoulíkia,O gátos,O peteinós,5.527542e-08,24.108787,-16.710938,5,0.015045,6.054582,-4.196716,4,0.017010,5.877503,-4.073975,4
6,swapped,null,match,pontíkia,O peteinós,O gátos,2.580992e-07,21.885571,-15.169922,4,0.033491,4.900091,-3.396484,4,0.004659,7.745867,-5.369026,4
7,swapped,null,neutral,fagitó,O peteinós,O gátos,1.135964e-03,9.781867,-6.780273,4,0.032153,4.958912,-3.437256,4,0.013623,6.197847,-4.296021,4
8,swapped,null,mismatch,skoulíkia,O peteinós,O gátos,2.446022e-07,21.963060,-15.223633,5,0.034640,4.851397,-3.362732,4,0.003056,8.354339,-5.790787,4
9,swapped,overt,match,pontíkia,O peteinós,O gátos,5.208658e-07,20.872585,-14.467773,4,0.039607,4.658116,-3.228760,4,0.011094,6.494054,-4.501335,4


## Note: antecedent-QA probe not used as a measure

We tested whether the model could directly indicate its preferred antecedent by scoring candidate answers to a comprehension question (e.g. "Piós épsachne gia pontíkia?", i.e., Who is looking for worms?).

Across all conditions (pronoun: null/overt, world-knowledge: match/neutral/mismatch), the model consistently assigned higher probability to whichever NP was mentioned most recently in the sentence (the 'object'), regardless of species or plausibility. To confirm this, we ran a control swapping which animal was NP1 vs NP2: the object-prefence effect remained after changing word order, tracking sentence position rather than world knowledge.

This indicates the probe is picking up a structural recency bias, not genuine coreference resolution, at least for this base (non-instruction-tuned) model. We're not using this measure going forward; the critical-word surprisal/predictability scores remain the main measure for the world-knowledge manipulation.

## Score the full stimuli set

In [11]:
test_cases = [
    {"pronoun": "null", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"pronoun": "null", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"pronoun": "null", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
    {"pronoun": "overt", "world_knowledge": "match", "critical_word": "pontíkia"},
    {"pronoun": "overt", "world_knowledge": "neutral", "critical_word": "fagitó"},
    {"pronoun": "overt", "world_knowledge": "mismatch", "critical_word": "skoulíkia"},
]

base_null = "O gátos íde ton peteinó ótan épsachne gia {word}."
base_overt = "O gátos íde ton peteinó ótan aftós épsachne gia {word}."

results3 = []
for case in test_cases:
    context = (base_null if case["pronoun"] == "null" else base_overt).format(word=case["critical_word"])
    question = f"Piós épsachne gia {case['critical_word']}?"
    candidates = ["O gátos", "O peteinós"]
    probs = score_antecedent(context, question, candidates, tokenizer, model, device)
    results3.append({
        "pronoun": case["pronoun"],
        "world_knowledge": case["world_knowledge"],
        "critical_word": case["critical_word"],
        "p_subject_gatos": probs["O gátos"],
        "p_object_peteinos": probs["O peteinós"],
    })

df3 = pd.DataFrame(results3)
df3

NameError: name 'score_antecedent' is not defined

In [4]:
stimuli = pd.read_csv("stimuli_demo.csv")
records = []
for _, row in stimuli.iterrows():
    scores = score_critical_word(row["full_sentence"], row["critical_word"], tokenizer, model, device)
    records.append({**row.to_dict(), **scores})

results_df = pd.DataFrame(records)
results_df[["item_id", "condition", "referent_bias", "critical_word", "probability", "surprisal_bits", "n_tokens"]]

FileNotFoundError: [Errno 2] No such file or directory: 'stimuli_demo.csv'

## Compare conditions within each item

For each item, the congruent conditions should show noticeably higher probability (lower surprisal) than a mismatched or neutral comparison. If probabilities look flat across conditions, that item's manipulation likely isn't landing and is worth revising or dropping.

In [ ]:
pivot = results_df.pivot_table(index="item_id", columns="condition", values="probability")
pivot

In [ ]:
# save results for downstream use (e.g. reading into your R wrangling project)
results_df.to_csv("output/scored_stimuli_demo.csv", index=False)